In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas_datareader as data

In [ ]:
plt.style.use('fivethirtyeight')
%matplotlib inline

# Importing Google dataset from yfinance

In [ ]:
import yfinance as yf
import datetime as dt

stock = "GOOG"
#start = dt.datetime(2000, 1, 1)
#end = dt.datetime(2025, 5, 1)

#df = yf.download(stock, start, end)

start = input("Enter start date (YYYY-MM-DD): ")
end = input("Enter end date (YYYY-MM-DD): ")

start_date = dt.datetime.strptime(start, "%Y-%m-%d")
end_date = dt.datetime.strptime(end, "%Y-%m-%d")

df = yf.download(stock, start=start_date, end=end_date)

In [ ]:
df.columns = [col[0] if col[1] == '' else f"{col[0]}_{col[1]}" for col in df.columns]
df.head()

# Pre Processing


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:

df = df.reset_index()

In [ ]:
df.columns

In [ ]:
df.to_csv("Google.csv")

In [ ]:
data01 = pd.read_csv("Google.csv")

In [ ]:
data01.head()

# Candlesticks 

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Candlestick(x = data01['Date'], open = data01['Open_GOOG'], 
                                    high = data01['High_GOOG'],
                                    low = data01['Low_GOOG'], 
                                    close = data01['Close_GOOG'])])
fig.update_layout(xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
df.head()

In [ ]:
df.set_index('Date', inplace=True)        # Make Date the index
plt.figure(figsize=(20, 6))
plt.plot(df['Close_GOOG'], label='Closing Price', linewidth=1)
plt.title('Closing prices over time')
plt.xlabel("Date")                      
plt.ylabel("Price") 
plt.legend()
plt.grid(True)                           
plt.show()

In [ ]:
plt.figure(figsize=(20, 6))
plt.plot(df['Open_GOOG'], label='Opening Price', linewidth=1)
plt.title('Opening prices over time')
plt.xlabel("Date")                        
plt.ylabel("Price")                       
plt.legend()
plt.grid(True)                           
plt.show()


In [ ]:
plt.figure(figsize=(20, 6))
plt.plot(df['High_GOOG'], label='High Price', linewidth=1)
plt.title('High prices over time')
plt.xlabel("Date")                        
plt.ylabel("Price")                       
plt.legend()
plt.grid(True)                           
plt.show()

In [ ]:
print(df[['Open_GOOG', 'High_GOOG', 'Close_GOOG', 'Volume_GOOG']].head(15))


In [ ]:
plt.figure(figsize=(20, 6))
plt.plot(df['Open_GOOG'], label='Open Price', linewidth='1')
plt.plot(df['High_GOOG'], label='High Price', linewidth='1')
plt.plot(df['Close_GOOG'], label='Close Price', linewidth='1')
plt.title('Open, High, and Close Prices Over Time')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(20, 6))         # sets the figure size to 20 inches wide and 6 inches tall
plt.plot(df['Volume_GOOG'], label='Volume', linewidth=1)
plt.title('Volume over time')
plt.xlabel("Date")                        
plt.ylabel("Volume")                       
plt.legend()
plt.grid(True)                           
plt.show()


# Moving Average

In [ ]:
# [10, 20, 30, 40, 50, 60, 70, 80, 90]
# moving average for last 5 days -> null null null null 30.0 40.0 50.0

temp_data = [10, 20, 30, 40, 50, 60, 70, 80, 90]
print(sum(temp_data[2:7])/5)

In [ ]:
import pandas as pd
df01 = pd.DataFrame(temp_data)

In [ ]:
df01.rolling(5).mean()

In [ ]:
ma100 = df.Close_GOOG.rolling(100).mean()
ma100

In [ ]:
ma200 = df.Close_GOOG.rolling(200).mean()
ma200

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df.Close_GOOG, label = f'{stock} Close Price', linewidth = 1)
plt.plot(ma100, label = f'{stock} Moving Average 100 Price', linewidth = 1)
plt.plot(ma200, label = f'{stock} Moving Average 200 Price', linewidth = 1)
plt.legend()
plt.grid(True)  
plt.show()

In [ ]:
ema100 = df.Close_GOOG.ewm(span=100, adjust = False).mean()

In [ ]:
ema200 = df['Close_GOOG'].ewm(span=200, adjust = False).mean()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df.Close_GOOG, label = f'{stock} Close Price', linewidth = 1)
plt.plot(ema100, label = f'{stock} Exp. Moving Average 100 Price', linewidth = 1)
plt.plot(ema200, label = f'{stock} Exp. Moving Average 200 Price', linewidth = 1)
plt.legend()
plt.grid(True)
plt.show()

# Training & Testing

In [ ]:
# dividing 70% to train data and 30% to test data.
data_training = pd.DataFrame(df['Close_GOOG'][0:int(len(df)*0.70)])
data_testing = pd.DataFrame(df['Close_GOOG'][int(len(df)*0.70): int(len(df))])

In [ ]:
data_training.shape

In [ ]:
data_testing.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range = (0, 1))

In [ ]:
data_training_array = scaler.fit_transform(data_training)

In [ ]:
data_training_array

In [ ]:
data_training_array.shape[0]  # 0 indicates rows

In [ ]:
x_train = []
y_train = []

for i in range(30, data_training_array.shape[0]):
    x_train.append(data_training_array[i-30:i])
    y_train.append(data_training_array[i, 0])

x_train, y_train  = np.array(x_train), np.array(y_train)

In [ ]:
x_train.shape

# Model Building

In [ ]:
from keras.layers import Dense, Dropout, LSTM
from keras.models import Sequential

In [ ]:
model = Sequential()

#default activation is tanh here instead of relu
model.add(LSTM(units = 50, return_sequences = True, input_shape = (30,1)))
model.add(Dropout(0.2))

model.add(LSTM(units = 60, return_sequences = True))
model.add(Dropout(0.3))

model.add(LSTM(units = 80, return_sequences = True))
model.add(Dropout(0.4))

model.add(LSTM(units = 120))
model.add(Dropout(0.5))

model.add(Dense(units = 1))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer = 'adam', loss = 'mean_squared_error')
model.fit(x_train, y_train, epochs = 50)

In [ ]:
past_30_days = data_training.tail(30)
past_30_days

In [ ]:
import pandas as pd

final_df = pd.concat([past_30_days, data_testing])

In [ ]:
final_df.head()

In [ ]:
input_data = scaler.fit_transform(final_df)

In [ ]:
x_test = []
y_test = []

for i in range(30, input_data.shape[0]):
    x_test.append(input_data[i-30:i])
    y_test.append(input_data[i, 0])

x_test, y_test  = np.array(x_test), np.array(y_test)

In [ ]:
x_test.shape

In [ ]:
y_predicted = model.predict(x_test)

In [ ]:
y_predicted = scaler.inverse_transform(y_predicted)
y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

In [ ]:
last_30_days = input_data[-30:]  # Take last 50 days

future_input = last_30_days.reshape(1, -1)[0].tolist()  # Flatten to list

next_7_days = []

for i in range(7):
    x_future = np.array(future_input[-30:]).reshape(1, 30, 1)  # Reshape to (1, 50, 1)
    pred = model.predict(x_future, verbose=0)[0][0]
    next_7_days.append(pred)
    future_input.append(pred)

# Inverse transform the predictions to original scale
next_7_days = scaler.inverse_transform(np.array(next_7_days).reshape(-1, 1))


# quetion:
"Why does my graph start from around October 2023, even though my dataset starts from May 2023?"
answer: The first 100 days (from May 2023 to around late August 2023) are used only for input, not for prediction.
So the first actual prediction corresponds to roughly day 101, which starts showing around October 2023 in your graph.

In [ ]:
test_data_len = len(y_test)
test_dates = df.index[-test_data_len:]  # Last N dates from index

from datetime import timedelta

last_date = test_dates[-1]
next_7_dates = [last_date + timedelta(days=i+1) for i in range(7)]



In [ ]:
print("Next 7 Days Forecast:")
for date, price in zip(next_7_dates, next_7_days):
    print(f"{date.strftime('%Y-%m-%d')}: {price[0]:.2f}")

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test_dates, y_test, label='Actual Price', linewidth=1)
plt.plot(test_dates, y_predicted, label='Predicted Price', linewidth=1)
plt.plot(next_7_dates, next_7_days, label='Next 7 Days Forecast', linestyle='dashed', linewidth=1.5, color='red')

plt.xlabel("Date")
plt.ylabel("Stock Price")
plt.title("Actual vs Predicted Stock Price + Next 7 Days Forecast")
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


# Evaluation of model : 

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Calculate metrics
mae = mean_absolute_error(y_test[:, 0], y_predicted[:, 0])
rmse = np.sqrt(mean_squared_error(y_test[:, 0], y_predicted[:, 0]))
r2 = r2_score(y_test[:, 0], y_predicted[:, 0])

# Print results
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")


In [ ]:
model.save('stock_dl_model.h5')